# 09c | Drills: A/B testing

Goes with **09a and 09b**: sample size and duration, analyzing an experiment dataset, relative lift, SRM, revenue metrics, CUPED, peeking, plus spoken questions on experiment design and interpretation.

**How to use this page:** There are no input boxes here. There are two kinds of problems:

- **Computation problems**: the code and its output are shown right below. First think about which function and formula to use, then read on for the answer;
- **Spoken problems** (marked 🗣 after the title): answer out loud first (30–60 seconds), then compare with the reference answer.

Problems on a page run in order, so variables and functions defined earlier can be reused later. Every problem assumes this has already been run:

```python
import numpy as np
import pandas as pd
from scipy import stats
```

### Problem 1 · The sample size formula

Baseline conversion rate 5%, and you want to detect an absolute lift of 0.5 percentage points (α = 0.05 two-sided, power = 0.8). How many users per group?

```python
def sample_size(p1, mde, alpha=0.05, power=0.8):
    p2 = p1 + mde
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    return (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / mde ** 2

round(sample_size(0.05, 0.005))
```

**Result:**

```text
31231
```

> The lower the baseline and the smaller the MDE, the more users you need. With conversion rates of a few percent, tens of thousands per group is normal.

### Problem 2 · Halve the MDE: how much bigger is the sample?

Same 5% baseline; sample size per group for MDEs of 0.25, 0.5, 1 and 2 percentage points.

```python
for mde in [0.0025, 0.005, 0.01, 0.02]:
    print(mde, round(sample_size(0.05, mde)))
```

**Result:**

```text
0.0025 122121
0.005 31231
0.01 8155
0.02 2209
```

> Each halving of the MDE roughly quadruples the sample size (not exactly 4×, because $p_2$ changes too).

### Problem 3 · How many days to run

4000 new users enter the experiment per day, split evenly between the groups. With the sample size from Problem 1, how many days do you need?

```python
n_per_group = sample_size(0.05, 0.005)
days = 2 * n_per_group / 4000
round(days, 1)
```

**Result:**

```text
15.6
```

> Rounded up, 16 days. The actual schedule also has to run in full weeks (so each day of the week appears equally often) and cover the metric's own observation window. So in practice you would usually schedule 3 weeks (21 days).

### Problem 4 · Sample size for a mean metric

The standard deviation of time spent per user is 20 minutes, and you want to detect a 1-minute difference. What do the exact formula and the shortcut $16\sigma^2/\delta^2$ give?

```python
sigma, delta = 20, 1
z_sum = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
exact = 2 * sigma ** 2 * z_sum ** 2 / delta ** 2
quick = 16 * sigma ** 2 / delta ** 2
print(round(exact), quick)
```

**Result:**

```text
6279 6400.0
```

> $2(1.96 + 0.84)^2 \approx 15.7$, so the shortcut uses 16. In an interview, getting the order of magnitude in your head is enough.

### Problem 5 · Analyzing an experiment dataset

8000 users per group; the true conversion rate is 8% in control and 9% in treatment. Compute the difference in conversion rates, the 95% confidence interval (unpooled standard error) and the p-value (pooled standard error).

```python
rng = np.random.default_rng(21)
n_c = n_t = 8000
conv_c = (rng.random(n_c) < 0.08).astype(int)
conv_t = (rng.random(n_t) < 0.09).astype(int)

p_c, p_t = conv_c.mean(), conv_t.mean()
diff = p_t - p_c
se = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
pooled = (conv_c.sum() + conv_t.sum()) / (n_c + n_t)
z = diff / np.sqrt(pooled * (1 - pooled) * (1 / n_c + 1 / n_t))
pval = 2 * stats.norm.sf(abs(z))
print(round(p_c, 4), round(p_t, 4), round(diff, 4))
print(round(diff - 1.96 * se, 4), round(diff + 1.96 * se, 4), round(pval, 4))
```

**Result:**

```text
0.0795 0.0859 0.0064
-0.0022 0.0149 0.1432
```

> The confidence interval uses each group's own variance (unpooled); the test of H₀ "the groups are the same" uses the pooled proportion.
>
> This time the result is **not significant**: the true difference is 1 percentage point, but with 8000 users per group the power is only about 60% ($z \approx 0.01/0.0044 \approx 2.26$, $\Phi(2.26 - 1.96) \approx 0.62$). Experiments with a real effect often come out non-significant, which is why you compute the sample size in advance.

### Problem 6 · Relative lift

Convert the previous result into a relative lift (how many percent higher treatment is than control), with an approximate interval.

```python
lift = diff / p_c
print(round(lift, 4), round((diff - 1.96 * se) / p_c, 4), round((diff + 1.96 * se) / p_c, 4))
```

**Result:**

```text
0.0802 -0.0272 0.1875
```

> This treats $p_c$ as a known constant, so it is only an approximation; the rigorous approach is the delta method or a bootstrap. When reporting, give both the absolute and the relative lift: "conversion rose by about 0.6 percentage points, a relative lift of about 8%, but the interval includes 0, so we can't yet be sure there is a lift".

### Problem 7 · SRM: imbalance and sample size

Planned 50/50. (1) 50400 vs. 49600; (2) 10080 vs. 9920. Both have the same proportional imbalance (50.4%). What is the SRM test p-value for each?

```python
print(round(stats.chisquare([50400, 49600]).pvalue, 4))
print(round(stats.chisquare([10080, 9920]).pvalue, 4))
```

**Result:**

```text
0.0114
0.2579
```

> The same 50.4% gives p ≈ 0.011 with the large sample and p ≈ 0.26 with the small one. The common warning threshold is 0.001; the larger the sample, the more suspicious the same imbalance.

### Problem 8 · Revenue: t-test plus bootstrap

Revenue is mostly 0, with a few large values. Compare revenue per user between the groups: the p-value of Welch's t-test, and a 95% bootstrap confidence interval for the difference.

```python
rev_c = conv_c * rng.lognormal(3.0, 1.2, n_c)
rev_t = conv_t * rng.lognormal(3.0, 1.2, n_t)
print(round(rev_t.mean() - rev_c.mean(), 3), round(stats.ttest_ind(rev_t, rev_c, equal_var=False).pvalue, 4))

boot = []
for _ in range(2000):
    bc = rng.choice(rev_c, size=n_c, replace=True)
    bt = rng.choice(rev_t, size=n_t, replace=True)
    boot.append(bt.mean() - bc.mean())
print(np.percentile(boot, [2.5, 97.5]).round(3))
```

**Result:**

```text
-0.12 0.7026
[-0.744  0.502]
```

> Resample each group **separately** with replacement. Revenue is mostly 0 with a few large values, so its variability relative to its mean (standard deviation ÷ mean) is far larger than for conversion rate; at the same sample size the power is much lower and the interval wide. That is exactly why the difference here is negative and far from significant: the difference in conversion is drowned out by revenue noise.

### Problem 9 · CUPED

The same metric measured before the experiment, `pre`, has a correlation of about 0.8 with the in-experiment metric `post`. How much does CUPED adjustment reduce the standard error of the treatment effect?

```python
rng = np.random.default_rng(22)
m = 10000
pre = rng.normal(100, 20, m)
group = rng.integers(0, 2, m)                         # 0 control, 1 treatment
post = 0.8 * pre + rng.normal(20, 12, m) + 2.0 * group

theta = ((pre - pre.mean()) * (post - post.mean())).mean() / pre.var()
adj = post - theta * (pre - pre.mean())

def effect_and_se(v):
    t, c = v[group == 1], v[group == 0]
    return t.mean() - c.mean(), np.sqrt(t.var(ddof=1) / len(t) + c.var(ddof=1) / len(c))

est0, se0 = effect_and_se(post)
est1, se1 = effect_and_se(adj)
print(round(theta, 3))
print("unadjusted:", round(est0, 3), round(se0, 3))
print("CUPED:", round(est1, 3), round(se1, 3))
```

**Result:**

```text
0.805
unadjusted: 1.926 0.397
CUPED: 1.795 0.238
```

> θ = Cov(pre, post)/Var(pre). Both effect estimates are around 2, but the standard error drops to about $\sqrt{1 - 0.8^2} = 0.6$ times the original, equivalent to about 2.8 times the sample size. `pre` is measured before the experiment and is independent of assignment, so the adjustment introduces no bias.

### Problem 10 · Peeking inflates false positives

1000 A/A experiments, 1000 users per group at the end, a look every 100 users (10 looks in total), declaring significance as soon as any look has p < 0.05. What is the false-positive rate?

```python
rng = np.random.default_rng(23)
n_sim, n_final, step = 1000, 1000, 100
a = rng.normal(size=(n_sim, n_final))
b = rng.normal(size=(n_sim, n_final))

ever_sig = np.zeros(n_sim, dtype=bool)
for k in range(step, n_final + 1, step):
    p_k = stats.ttest_ind(a[:, :k], b[:, :k], axis=1).pvalue
    ever_sig = ever_sig | (p_k < 0.05)
final_sig = stats.ttest_ind(a, b, axis=1).pvalue < 0.05
print(round(final_sig.mean(), 3), round(ever_sig.mean(), 3))
```

**Result:**

```text
0.055 0.183
```

> Looking only at the end gives about 5%; looking 10 times and stopping when significant pushes the false-positive rate to about 18%. The more looks, the worse it gets.

### Problem 11 · Design question: checkout redesign 🗣

An e-commerce company wants to shrink checkout from three steps to one, expecting a higher order rate. Walk through the experiment design.

**Reference answer:**

**Goal and decision:** does one-step checkout raise the share of completed purchases, to decide whether to launch to everyone.

**Metrics:** the primary metric is "share of users reaching the checkout page who complete payment" (user-level). Guardrails: payment failure rate, refund / cancellation rate, average order value, page load time, customer-service complaints.

**Randomization:** by user, so the same user always sees the same version; only users who reach the checkout page count toward the experiment (triggered analysis), with the same trigger condition in both groups.

**Sample size and duration:** from the baseline completion rate, the MDE (e.g. a 2% relative lift), α = 0.05 and power = 0.8, compute users per group, then divide by the number of users reaching checkout per day; run at least two weeks to cover weekends and paydays.

**Checks and analysis:** run SRM and A/A checks first; compare the difference in completion rates with its confidence interval; check consistency across new vs. existing users and mobile vs. desktop (segments declared in advance).

**Decision:** launch only if the primary metric improves significantly and no guardrail worsens. Watch the refund rate in particular: one-step checkout may increase impulse orders, followed by more refunds.

### Problem 12 · Choosing the primary metric 🗣

For a redesign of search result ranking, is "search click-through rate" a good primary metric?

**Reference answer:**

Not really. Click-through rate is easy to inflate with clickbait, and more clicks don't mean people found what they wanted; it is also a ratio metric whose denominator (number of searches) can itself be affected by the treatment (users search again and again when results are poor).

Better choices: a user-level "search success rate" (a purchase / meaningful dwell within some time after searching, without immediately searching again), or purchase conversion per user. Click-through rate can be a secondary metric, with "zero-result rate" and "re-search rate" as guardrails.

Principle: the primary metric should align with the long-term goal, be sensitive enough to the change, be hard to game with short-term behavior, and ideally be user-level.

### Problem 13 · How to set the MDE 🗣

When computing a sample size, how should you choose the MDE?

**Reference answer:**

The MDE is not "my guess at the effect size" but "**how large an effect would be worth launching**": set it from the cost and benefit of the change, e.g. the benefit must at least cover engineering and maintenance costs, which translates into needing a 1% lift in conversion.

Then check practical constraints: is the duration implied by this MDE acceptable? If it would take six months, either accept a larger MDE or find ways to reduce variance (CUPED, a more sensitive metric). The effect sizes of similar past experiments are also a useful reference; most changes have very small true effects.

### Problem 14 · After finding an SRM 🗣

At the end of an experiment the groups are 50.6% vs. 49.4%, with an SRM test p < 0.0001. What next?

**Reference answer:**

**Don't look** at the metric results yet: the split can no longer be trusted, and any difference could come from "who got logged" rather than from the treatment itself.

Investigate: (1) look at the split by day to find when the imbalance started (it may coincide with a release); (2) break it down by platform, browser, country, new vs. existing users to locate which users are missing; (3) check whether the treatment group crashes or loads slowly more often, causing lost logs; (4) check whether bot filtering, redirects or trigger conditions treat the groups asymmetrically.

Once the cause is found and fixed, rerun the experiment. Only consider using the existing data if you can confirm the problem is unrelated to the outcome and can cleanly remove the affected portion.

### Problem 15 · Novelty effect 🗣

A new feature looked great in its first week and noticeably weaker by the third week. What might be going on? How do you tell?

**Reference answer:**

Possibly a **novelty effect**: users click more because something is new and drop back once they're used to it. Conversely, a change that requires learning can start worse and then improve (change aversion / learning effect).

How to tell: (1) plot the treatment effect over time (by day or week) and see whether it decays; (2) align by "days since the user first saw the new feature" rather than calendar day; (3) compare new and existing users: new users have no "old version" to compare with and are not subject to the novelty effect, so their effect is closer to the long-term effect.

Remedies: run the experiment longer and decide based on the stabilized effect; if needed, keep a long-term holdout group.

### Problem 16 · A non-significant result: what to tell the PM 🗣

The experiment gives p = 0.25, and the PM asks: "So that means the feature doesn't work?" How do you answer?

**Reference answer:**

"We can't say that. Not significant means this experiment's data can't distinguish 'has an effect' from 'has no effect'." Then look at the confidence interval:

- If the interval is (−0.2%, +0.8%) and the smallest lift we care about is 0.5%, a meaningful effect **can't be ruled out yet**; the experiment may be underpowered, and we could consider running longer or with more traffic;
- If the interval is (−0.1%, +0.1%), any meaningful effect has been ruled out, and we can say with some confidence that "even if there's a lift, it's too small to be worth it".

So report the interval, not just "not significant".

### Problem 17 · Two-sided marketplace: a delivery-fee experiment 🗣

A food-delivery platform wants to test the effect of "lower delivery fees" on order volume. What's wrong with simply randomizing by user?

**Reference answer:**

**Interference**: treatment users order more because of the lower fee, which ties up courier capacity, so control users get longer delivery times and order less. The groups are no longer independent, and a user-level comparison **overstates** the effect (the treatment group takes resources from the control group); the true effect after a full launch would be much smaller.

Remedies: randomize by region (city / delivery zone), or rotate over time (switchback: the same zone uses the new price in some time slots and the old price in others). The price is fewer randomization units and higher variance, so it takes longer, and standard errors must be clustered by region or time slot in the analysis.

### Problem 18 · The unit of analysis for CTR 🗣

Users are randomized into groups, and you compare click-through rates (total clicks ÷ total views). What's wrong with treating each view as an independent sample in a two-proportion z-test?

**Reference answer:**

Multiple views from the same user are correlated (some people click a lot, others don't), so they are not independent samples. Treating views as independent **underestimates the variance**: standard errors are too small, p-values too small, and false positives more frequent.

The principle is that **the unit of analysis must match the unit of randomization**. Options: compute the standard error of the ratio "total clicks / total views" with the delta method; or bootstrap by user (each resample draws users, along with all their views and clicks); or switch to a user-level metric, such as averaging each user's own click-through rate (but that means something different from overall CTR, which you should state).

### Problem 19 · Primary metric up, guardrail worse 🗣

The new version raises conversion significantly by 3%, but page load time (a guardrail) got 10% slower. Do you launch?

**Reference answer:**

You can't look only at the primary metric. First find out: (1) whether the slowdown is a fixable engineering issue (e.g. missing caching), and whether conversion would be even higher once it's fixed; (2) the slowdown's effect on long-term metrics (retention, repeat purchases) may not show up in this experiment; what does historical data say about load time and retention? (3) what threshold was set for the guardrail in advance, and does 10% cross it?

A common conclusion: don't launch to everyone yet; fix the performance issue and run another round; or launch conditionally within acceptable bounds while monitoring continuously. The key is to lay out the trade-off clearly for the decision-makers rather than just reporting a p-value.

### Problem 20 · A significant result in a segment 🗣

The overall effect is not significant, but an analyst finds p = 0.01 in the "Android, under 25" segment and suggests launching only for that group. What do you think?

**Reference answer:**

Be careful. If you search after the fact across many segments (platform × age × region…), you will always hit a few with p < 0.05; this is a multiple-testing problem, and this p = 0.01 may well be chance.

What to do: (1) ask whether this segment was specified in advance; (2) if it was found after the fact, treat it as a **hypothesis** and verify it with a new experiment targeting just that group; (3) check whether the effect points the same way in neighboring segments (e.g. a similar trend for 25–30-year-olds) and whether there is a plausible product explanation; (4) at minimum, apply a multiple-testing correction.

## Recap

| Task | Code / key point |
|---|---|
| Sample size for two proportions | $(z_{1-\alpha/2} + z_{1-\beta})^2 [p_1(1-p_1) + p_2(1-p_2)] / \mathrm{MDE}^2$ |
| Sample size shortcut for a mean | $16\sigma^2/\delta^2$ (per group) |
| Duration | sample size ÷ daily traffic, at least 1–2 full weeks |
| Interval / test for the difference | unpooled SE / pooled SE |
| Relative lift | `diff / p_c`; report both absolute and relative |
| SRM | `stats.chisquare([n_a, n_b]).pvalue`, warning threshold 0.001 |
| Heavy-tailed metrics | Welch t + per-group bootstrap |
| CUPED | `post - theta * (pre - pre.mean())`, variance multiplied by $1 - \rho^2$ |
| Peeking | stopping after k looks pushes the false-positive rate far above α |